In [4]:
import re
from pathlib import Path
import pandas as pd
import numpy as np

RAW_FILE = Path("raw_dataset/standardized_caafimaad_master.xlsx")
OUTPUT_REVIEW_FILE = Path("clean_dataset_for_review/caafimaad_cleaned_for_review.xlsx")
OUTPUT_FINAL_FILE = Path("ready_dataset/caafimaad_model_ready.csv")

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_rows", 100)

## 1. Load the Excel file

We keep the raw file unchanged. All cleaning results are saved into new files.

In [5]:
df = pd.read_excel(RAW_FILE)
print(df.shape)
df.head()

(1144, 5)


,url,headline,body,category,source
0,https://www.bbc.com/somali/articles/cg7ejn4gx07o,Daraasad: Uurka labaad wuxuu 'si gaar ah' u beddelaa maskaxda haweenka,"Xigashada Sawirka,Getty Images\nUuurka labaad ayaa sababa in maskaxda hooyada ay siyaabo ""gaar ah"" isu beddesho kuwaasoo ka caawin kara inay diiradda saaraa...",saynis_iyo_caafimaad,BBC Somali
1,https://www.bbc.com/somali/articles/c705rww15e3o,Dayax-gacmeed bani'aadam sida oo 50 sano ka dib dayaxa u kicitimay,"Xigashada Sawirka,Getty Images\nIn ka badan konton sano ka dib, NASA hawshii ugu horeysay ee Dayaxa ayey dib u billowday.\nGantaalka loogu talagalay howlgal...",saynis_iyo_caafimaad,BBC Somali
2,https://www.bbc.com/somali/articles/cly2y5kg3r0o,Muhiimadda habeenka 29-aad ee Ramadaan,"Xigashada Sawirka,Getty Images\nSida habeenada kale ee Ramadaanka, habeenka 29-aad ee bisha barakaysan waa mid ka mid ah habeenada sida gaarka ah looga sugo...",saynis_iyo_caafimaad,BBC Somali
3,https://www.bbc.com/somali/articles/c0lj6935l68o,Muxuu yahay dhagaxa qoslaya ee laga helay Ingiriiska?,"Xigashada Sawirka,Tony Jolliffe/BBC\nChristine Clark, oo 64 jir ah, ayaa raadinaysay walxo qadiimi ah maalin ka dib ciidii Christmas-ka iyada oo lugaynaysay...",saynis_iyo_caafimaad,BBC Somali
4,https://www.bbc.com/somali/articles/cg5n002r7l0o,Maxaad ka taqanaa nuucyada suxuurta ugu caansan Afrika?,"Xigashada Sawirka,Getty Images\nInta lagu jiro bisha barakeysan ee Ramadan, waxaa kordha rabitaanka cunista cuntooyin kala duwan. Dad badanna waxay tixgeliy...",saynis_iyo_caafimaad,BBC Somali


## 2. Basic structure checks

In [7]:
expected_cols = ["url", "headline", "body", "category", "source"]
print("Columns:", df.columns.tolist())
missing_cols = [c for c in expected_cols if c not in df.columns]
print("Missing expected columns:", missing_cols)

for col in expected_cols:
    if col in df.columns:
        print(f"{col}: missing =", df[col].isna().sum())

print("Category values:")
print(df["category"].value_counts(dropna=False))

print("Source values:")
print(df["source"].value_counts(dropna=False))

Columns: ['url', 'headline', 'body', 'category', 'source']
Missing expected columns: []
url: missing = 0
headline: missing = 0
body: missing = 0
category: missing = 0
source: missing = 0
Category values:
category
Caafimaad               646
saynis_iyo_caafimaad    498
Name: count, dtype: int64
Source values:
source
Goobjoog      646
BBC Somali    498
Name: count, dtype: int64


## 3. Standardize text fields and category labels

For this category file, all category values should become exactly:

```text
caafimaad
```

In [31]:
def normalize_basic_text(text):
    if pd.isna(text):
        return ""
    
    text = str(text)

    # Important: handle both real newlines and literal "\n"
    text = text.replace("\\n", "\n")
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    text = text.replace("\u00a0", " ")
    text = text.replace("\ufeff", "")
    text = text.replace("“", '"').replace("”", '"')
    text = text.replace("‘", "'").replace("’", "'")
    text = text.replace("–", "-").replace("—", "-")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

df = df.copy()
for col in ["url", "headline", "body", "category", "source"]:
    df[col] = df[col].apply(normalize_basic_text)

df["source"] = df["source"].str.strip()
df["source"] = df["source"].replace({
    "BBC": "BBC Somali",
    "bbc somali": "BBC Somali",
    "goobjoog": "Goobjoog",
})

df["category_original"] = df["category"]
df["category"] = "caafimaad"

df[["source", "category_original", "category"]].drop_duplicates().head(20)

,source,category_original,category
0,BBC Somali,caafimaad,caafimaad
498,Goobjoog,caafimaad,caafimaad


## 4. Inspect body and headline lengths

These checks help us find very short articles, extremely long articles, or weak headlines.

In [32]:
df["body_word_count_raw"] = df["body"].str.split().str.len()
df["headline_word_count"] = df["headline"].str.split().str.len()

print("Body word count:")
print(df["body_word_count_raw"].describe(percentiles=[.05, .25, .5, .75, .95]))

print("Headline word count:")
print(df["headline_word_count"].describe(percentiles=[.05, .25, .5, .75, .95]))

Body word count:
count    1144.000000
mean      433.896853
std       335.880501
min        12.000000
5%         95.000000
25%       151.750000
50%       293.500000
75%       668.000000
95%      1016.500000
max      2670.000000
Name: body_word_count_raw, dtype: float64
Headline word count:
count    1144.000000
mean       11.140734
std         3.105937
min         3.000000
5%          7.000000
25%         9.000000
50%        11.000000
75%        13.000000
95%        17.000000
max        25.000000
Name: headline_word_count, dtype: float64


In [10]:
short_articles = df[df["body_word_count_raw"] < 50][["source", "headline", "body_word_count_raw", "body", "url"]]
short_articles

,source,headline,body_word_count_raw,body,url
351,BBC Somali,Daawo: Sida gabadhaan Soomaaliyeed uga badbaaday kansarka naasaha,37,Haboon ayaa hadda haweenka ka wacyi galisa kansarka una shegta inaysan ceeb dareemin oo ay la tacaalaan.\n©2026 BBC. BBC masuul kama ahan macluumadka bogagg...,https://www.bbc.com/somali/articles/cpw584ny9jwo
517,Goobjoog,Wasaaradda Caafimaadka oo Sharaxday Heerka Diyaargarowga Coronaviruses,40,Wasaaradda caafimaadka iyo daryeelka bulshada xukuumadda federaalka Soomaaliya ayaa maanta war ka soo saartay xaaladda cudurka Coronaviruses oo kiisaska ku ...,https://goobjoog.com/2020/03/02/wasaaradda-caafimaadka-oo-sharaxday-heerka-diyaargarowga-coronaviruses/
886,Goobjoog,"Fowsiyo Abiikar Nuur "" Nafaqo-darro Xun Ayaa Dalka Ku Haysata Caruurta Iyo Haweenka Soomaaliyeed""",12,Halkan Ka Daawo Xogta Ka Soo Baxday Wasaarada Caafimaadka Soomaaliya\n\nGoobjoog News,https://goobjoog.com/2020/08/25/fowsiyo-abiikar-nuur-nafaqo-darro-xun-ayaa-dalka-ku-haysata-caruurta-iyo-haweenka-soomaaliyeed/
891,Goobjoog,Xisbiga Wadeni Ee Somaliland oo Ka Tacsiyeey Geerida Qoraa Baashe Xaaji Xasan,31,"Xisbiga waddani Ee Somaliland ayaa dhambaal Tacsiya udiray ehalada, qaraabada iyo guud ahaan umada Soomaaliyeed ee uu ka baxay qoraa maxamed Baashe Xaaji Xa...",https://goobjoog.com/2020/06/18/xisbiga-wadeni-ee-somaliland-oo-ka-tacsiyeey-geerida-qoraa-baashe-xaaji-xasan/


## 5. Source-specific noise patterns

From this file, the repeated noise is different by source:

BBC Somali examples:
- `Xigashada Sawirka,...`
- `End of content`
- `End of Ugu akhris badan`
- `Warbixinada qotada dheer...`
- `Halkaan kaga soo biir`
- `Dhamaadka xayeysiinta`
- `©2026 BBC...`

Goobjoog examples:
- `Goobjoog News`
- `Googjoog News`
- `Dhageyso`
- `Halkaan hoose ka dhageyso:`
- `Halkaan ka Akhriso`

In [33]:
def line_frequency(dataframe, source_name, top_n=30):
    lines = []
    subset = dataframe[dataframe["source"] == source_name]
    for body in subset["body"]:
        for line in str(body).splitlines():
            line = normalize_basic_text(line)
            if line:
                lines.append(line)
    return pd.Series(lines).value_counts().head(top_n)

print("BBC repeated lines:")
display(line_frequency(df, "BBC Somali", 25))

print("Goobjoog repeated lines:")
display(line_frequency(df, "Goobjoog", 25))

BBC repeated lines:


Xigashada Sawirka,Getty Images                                                                                                                                         961
©2026 BBC. BBC masuul kama ahan macluumadka bogagga kale ee dibadda.Akhri xogta ku saabsan sida aan u abaarno bogagga dibadda.                                         498
End of Ugu akhris badan                                                                                                                                                485
End of content                                                                                                                                                         450
Warbixinada qotada dheer iyo wararka BBC Somali oo toos kuugu imanaaya WhatsApp.                                                                                       380
Halkaan kaga soo biir                                                                                                                            

Goobjoog repeated lines:


Goobjoog News                                                                                                                                                                                                                                                                                 478
Wasiirka Caafimaadka iyo Daryeelka Bulshada Xukuumadda Federaalka Soomaaliya ayaa bulshada ugu baaqday in lagu dadaalo kahor tagga cudurka si loo yareeyo saameynta uu bulshada ku yeelanayo.                                                                                                   7
Dhageyso                                                                                                                                                                                                                                                                                        7
Warkaan wixii kusoo kordha kala soco wararkeena kale                                                                              

## 6. Cleaning functions

The goal is **light cleaning**, not changing the meaning of Somali text.

Do not remove stopwords, do not stem words, and do not translate text.

In [34]:
BBC_REMOVE_LINE_PATTERNS = [
    r"^Xigashada Sawirka\s*,?.*$",   # removes: Xigashada Sawirka,Getty Images
    r"^Qoraalka Sawirka\s*,?.*$",
    r"^End of content$",
    r"^End of Ugu akhris badan$",
    r"^Dhamaadka xayeysiinta$",
    r"^Halkaan kaga soo biir$",
    r"^Warbixinada qotada dheer.*$",
    r"^©\d{4}\s+BBC\..*$",
    r"^BBC masuul kama ahan.*$",
    r"^Calaamadaha:$",
]

GOOBJOOG_REMOVE_LINE_PATTERNS = [
    r"^Goobjoog News$",
    r"^Googjoog News$",
    r"^Dhageyso$",
    r"^Halkaan hoose ka dhageyso:?$",
    r"^Halkaan ka Akhriso$",
    r"^Halkaan Ka Akhriso$",
    r"^Warkaan wixii kusoo kordha kala soco wararkeena kale$",
]

def remove_noise_lines(text, source):
    text = normalize_basic_text(text)

    patterns = []
    if source == "BBC Somali":
        patterns = BBC_REMOVE_LINE_PATTERNS
    elif source == "Goobjoog":
        patterns = GOOBJOOG_REMOVE_LINE_PATTERNS

    cleaned_lines = []

    for line in text.splitlines():
        line = normalize_basic_text(line)

        if not line:
            continue

        remove = any(
            re.search(pattern, line, flags=re.IGNORECASE)
            for pattern in patterns
        )

        if not remove:
            cleaned_lines.append(line)

    cleaned = "\n".join(cleaned_lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned)
    cleaned = re.sub(r"[ \t]+", " ", cleaned)

    return cleaned.strip()

def clean_headline(text):
    text = normalize_basic_text(text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["body_clean"] = df.apply(lambda row: remove_noise_lines(row["body"], row["source"]), axis=1)
df["headline_clean"] = df["headline"].apply(clean_headline)

df["body_word_count_clean"] = df["body_clean"].str.split().str.len()

In [35]:
df["body"][4]

'Xigashada Sawirka,Getty Images\nInta lagu jiro bisha barakeysan ee Ramadan, waxaa kordha rabitaanka cunista cuntooyin kala duwan. Dad badanna waxay tixgeliyaan faa\'iidada caafimaad iyo sidoo kale xaaladda cimilada marka ay dooranayaan waxa ay ku afuraan ama ku sahuraan.\nXilli ay qoysasku ka fikirayaan nooca cunto ee ku habboon afurka iyo saxuurta maalmaha bisha soonka, ayaa BBC-du waxa ay la hadashay qaar ka mid ah wariayaasheeda ku sugan dalal kala duwan ee Afrika si aan u ogaanno cuntooyinka inta badan laga cuno dalalkooda marka la gaadho suxuurta iyo afurka bisha Ramadan.\nEnd of content\n"Marka la gaadho afurka, waxaan ku billownaa timir iyo caano. kaddibna waxaan cunnaa waxa aan ugu yeedhno harira oo ah nooc maraq u eg oo dhadhan fiican leh, sidoo kalena kordhiya tamarta islamarkaana diiriya caloosha," ayay tiri wariyaha BBC-da, Basma El Atti.\nDhanka suxuurta, "badanaa waxaan cunnaa waxa loo yaqaan Sellou - waa cunto laga sameeyo bur iyo laws, waxaana sidoo kale cabnaa biyo ba

## 7. Compare before/after cleaning

In [36]:
print("Raw body length:")
print(df["body_word_count_raw"].describe(percentiles=[.05, .25, .5, .75, .95]))

print("Clean body length:")
print(df["body_word_count_clean"].describe(percentiles=[.05, .25, .5, .75, .95]))

Raw body length:
count    1144.000000
mean      433.896853
std       335.880501
min        12.000000
5%         95.000000
25%       151.750000
50%       293.500000
75%       668.000000
95%      1016.500000
max      2670.000000
Name: body_word_count_raw, dtype: float64
Clean body length:
count    1144.000000
mean      411.161713
std       315.076462
min        10.000000
5%         93.000000
25%       150.000000
50%       291.500000
75%       618.250000
95%       961.950000
max      2612.000000
Name: body_word_count_clean, dtype: float64


In [37]:
sample_cols = ["source", "headline_clean", "body", "body_clean"]
df.sample(10, random_state=42)[sample_cols]

,source,headline_clean,body,body_clean
218,BBC Somali,Daanyeerrada dhirta isku daweeya oo gacan ka geysan kara helidda daawo cusub,"Xigashada Sawirka,Getty Images\nNooc daanyeerrada ka mid ah ee gorillaha ayaa laga yaabaa inay aadmiga feker ka siiyaan in mustaqbalka lasoo saaro dawo, sid...","Nooc daanyeerrada ka mid ah ee gorillaha ayaa laga yaabaa inay aadmiga feker ka siiyaan in mustaqbalka lasoo saaro dawo, sida ay sheegeen saynisyahannada.\n..."
809,Goobjoog,Daraasad: Dadka Sida Tartiibta Ah Wax U Cuno Ma Cayilaan,Daraasad cusub ayaa sheegeysa in dadka cuntada tartiib u cuna ay yihiin kuwa aan cayilin. sidoo kale dadkaani kuma dhacaan cudurada loo yaqaanoMetabolic Syn...,Daraasad cusub ayaa sheegeysa in dadka cuntada tartiib u cuna ay yihiin kuwa aan cayilin. sidoo kale dadkaani kuma dhacaan cudurada loo yaqaanoMetabolic Syn...
501,Goobjoog,Maxaa Jirkaaga Ku Dhacaya Haddii Muddo Bil ah Aad Iska Dhaafto Sonkorta?,"Warbixin lagu soo daabacay wargeysa ""hemsleyandhemsley"" ayaa lagu sheegay in faa'iidooyin farabadan oo caafimaad ay leedahay haddii muddo bil ah aad iska jo...","Warbixin lagu soo daabacay wargeysa ""hemsleyandhemsley"" ayaa lagu sheegay in faa'iidooyin farabadan oo caafimaad ay leedahay haddii muddo bil ah aad iska jo..."
649,Goobjoog,Tirada Xaaladaha COVID19 Ee Dalka Laga Diiwaan Gelinayo Oo Hoos u Dhacay,"La taliyaha sare ee COVID19, kala taliyo wasaaradda caafimaadka Dr. Maxamed Maxamuud Cali (Fuje) ayaa warbixinta maalinlaha ah ee COVID19 ku sheegay in 24ki...","La taliyaha sare ee COVID19, kala taliyo wasaaradda caafimaadka Dr. Maxamed Maxamuud Cali (Fuje) ayaa warbixinta maalinlaha ah ee COVID19 ku sheegay in 24ki..."
323,BBC Somali,Waa tee da'da ugu habboon ragga ee ay ilmo fiican ku dhali karaan,"Xigashada Sawirka,Getty Images\nWaxaan ognahay in uu jiro xilli ay haweeneydu ka istaagto caadada, taasoo saamayn ku yeelata dhalmadeeda, laakin waxaa dhici...","Waxaan ognahay in uu jiro xilli ay haweeneydu ka istaagto caadada, taasoo saamayn ku yeelata dhalmadeeda, laakin waxaa dhici karta in aan si fiican loo eegi..."
506,Goobjoog,Ra'iisulwasaare Xamse oo Xariigjray Dhismaha Wasaaradda Caafimaadka Jubbaland,Ra'iisul Wasaaraha Xukuumadda Federaalka Soomaaliya Mudane Xamsa Cabdi Barre oo uu wehelinayo Madaxweynaha Dowlad-goboleedka Jubbaland Mudane Axmed Maxamed ...,Ra'iisul Wasaaraha Xukuumadda Federaalka Soomaaliya Mudane Xamsa Cabdi Barre oo uu wehelinayo Madaxweynaha Dowlad-goboleedka Jubbaland Mudane Axmed Maxamed ...
1005,Goobjoog,Maamulka gobolka Banaadir oo gubay raashin iyo daawooyin dhacay,Maamulka Gobolka banaadir ee Dowladda Soomaaliya ayaa maanta gubay raashin iyo Daawooyin la sheegay inay dhaceen.\nGuddoomiyaha maamulka degmada kaaraan Axm...,Maamulka Gobolka banaadir ee Dowladda Soomaaliya ayaa maanta gubay raashin iyo Daawooyin la sheegay inay dhaceen.\nGuddoomiyaha maamulka degmada kaaraan Axm...
107,BBC Somali,Riyada oo lagu biyo-baxo ma calaamad ayay u tahay dhalmo la'aanta?,"Xigashada Sawirka,Getty Images\nMarka ninku riyoodo (biyo baxo) inta uu hurdada ku jiro, khubaradu waxay xaaladdan ku tilmaamaan ""riyo baraarug leh"" (lucid ...","Marka ninku riyoodo (biyo baxo) inta uu hurdada ku jiro, khubaradu waxay xaaladdan ku tilmaamaan ""riyo baraarug leh"" (lucid dream). Rag badan way ka welwela..."
731,Goobjoog,Isbitaalka Guud Ee Garoowe Oo Qarka U Saaran In Uu Xirmo,Isbitaalka Guud ee Magaalada Garoowe ayaa lagu soo waramayaa in uu qarka u saaran yahay gabi ahaanba inuu xirmo.\nDhaqaatiirta iyo shaqaalaha Isbitalaka Guu...,Isbitaalka Guud ee Magaalada Garoowe ayaa lagu soo waramayaa in uu qarka u saaran yahay gabi ahaanba inuu xirmo.\nDhaqaatiirta iyo shaqaalaha Isbitalaka Guu...
170,BBC Somali,Sameecadaha dhegaha ee dhawaqa xanniba miyaa sababay dhibaatada maqalka ee dhalinyarada?,"Xigashada Sawirka,Getty Images\nHaddii ay ahaan lahayd dhawaqa yar ee kasoo baxa mashiinnada ay lacagta ku xisaabiyaan shaqaalaha dukaamada waaweyn amaba ka...",Haddii ay ahaan lahayd dhawaqa yar ee kasoo baxa ma

In [38]:
bbc_remaining_noise = df[
    (df["source"] == "BBC Somali") &
    (df["body_clean"].str.contains(r"(?m)^Xigashada Sawirka", case=False, regex=True, na=False))
]

print("BBC rows still containing image credit:", len(bbc_remaining_noise))

bbc_remaining_noise[["headline_clean", "body_clean"]].head()

BBC rows still containing image credit: 0


,headline_clean,body_clean


## 8. Duplicate detection

Remove exact duplicates first. Near duplicates should usually be reviewed before deleting.

In [39]:
def normalize_for_duplicate(text):
    text = normalize_basic_text(text).lower()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df["url_norm"] = df["url"].str.lower().str.strip()
df["headline_norm"] = df["headline_clean"].apply(normalize_for_duplicate)
df["body_norm"] = df["body_clean"].apply(normalize_for_duplicate)

print("Duplicate URLs:", df.duplicated("url_norm").sum())
print("Duplicate clean headlines:", df.duplicated("headline_norm").sum())
print("Duplicate clean bodies:", df.duplicated("body_norm").sum())

df[df.duplicated("headline_norm", keep=False)].sort_values("headline_norm")[["source", "headline_clean", "url"]].head(50)

Duplicate URLs: 0
Duplicate clean headlines: 17
Duplicate clean bodies: 3


,source,headline_clean,url
156,BBC Somali,Ramadan 2025: Wax ka ogow xubnaha muhiimka ah ee jirkaaga oo soonku faa'iido gaar ah u leeyahay,https://www.bbc.com/somali/articles/cxxz9g16yrxo
166,BBC Somali,Ramadan 2025: Wax ka ogow xubnaha muhiimka ah ee jirkaaga oo soonku faa'iido gaar ah u leeyahay,https://www.bbc.com/somali/articles/clyzle6rzepo
1094,Goobjoog,Shil Khasaare Geystay oo Ka Dhacay Muqdisho,https://goobjoog.com/2020/06/17/shil-khasaare-geystay-oo-ka-dhacay-muqdisho/
572,Goobjoog,Shil Khasaare Geystay oo Ka Dhacay Muqdisho,https://goobjoog.com/2020/08/08/shil-khasaare-geystay-oo-ka-dhacay-muqdisho-2/
40,BBC Somali,Talooyin ku anfacaya Ramadaanka si aad u ilaaliso caafimaadkaaga,https://www.bbc.com/somali/articles/cy59zn7w0zno
151,BBC Somali,Talooyin ku anfacaya Ramadaanka si aad u ilaaliso caafimaadkaaga,https://www.bbc.com/somali/articles/c72g0049rqzo
1066,Goobjoog,Wasaaradda Caafimaadka oo Ka Warbixisay Kororka Coronavirus 24-kii Saac Ee Tagtay Gudaha Dalka,https://goobjoog.com/2020/05/11/wasaaradda-caafimaadka-oo-ka-warbixisay-kororka-coronavirus-24-kii-saac-ee-tagtay-gudaha-dalka-4/
990,Goobjoog,Wasaaradda Caafimaadka oo Ka Warbixisay Kororka Coronavirus 24-kii Saac Ee Tagtay Gudaha Dalka,https://goobjoog.com/2020/05/05/wasaaradda-caafimaadka-oo-ka-warbixisay-kororka-coronavirus-24-kii-saac-ee-tagtay-gudaha-dalka-2/
914,Goobjoog,Wasaaradda Caafimaadka oo Ka Warbixisay Kororka Coronavirus 24-kii Saac Ee Tagtay Gudaha Dalka,https://goobjoog.com/2020/05/21/wasaaradda-caafimaadka-oo-ka-warbixisay-kororka-coronavirus-24-kii-saac-ee-tagtay-gudaha-dalka-8/
785,Goobjoog,Wasaaradda Caafimaadka oo Ka Warbixisay Kororka Coronavirus 24-kii Saac Ee Tagtay Gudaha Dalka,https://goobjoog.com/2020/05/20/wasaaradda-caafimaadka-oo-ka-warbixisay-kororka-coronavirus-24-kii-saac-ee-tagtay-gudaha-dalka-7/


In [40]:
# Keep first URL/body exact duplicate only. Do NOT remove duplicate headlines automatically, because daily health updates can have similar headlines.
df_no_exact_dupes = df.drop_duplicates(subset=["url_norm"]).copy()
df_no_exact_dupes = df_no_exact_dupes.drop_duplicates(subset=["body_norm"]).copy()
print(df.shape, "->", df_no_exact_dupes.shape)

(1144, 14) -> (1141, 14)


In [41]:
clean_df = df_no_exact_dupes.copy()

clean_df["flag_short_body"] = clean_df["body_word_count_clean"] < 50
clean_df["flag_very_short_headline"] = clean_df["headline_clean"].str.split().str.len() < 3
clean_df["flag_empty_body"] = clean_df["body_clean"].str.len() == 0
clean_df["flag_empty_headline"] = clean_df["headline_clean"].str.len() == 0

flag_cols = ["flag_short_body", "flag_very_short_headline", "flag_empty_body", "flag_empty_headline"]
clean_df[flag_cols].sum()

flag_short_body             7
flag_very_short_headline    0
flag_empty_body             0
flag_empty_headline         0
dtype: int64

In [42]:
clean_df[clean_df[flag_cols].any(axis=1)][[
    "source", "headline_clean", "body_word_count_clean", "body_clean", "url"
]].head(100)

,source,headline_clean,body_word_count_clean,body_clean,url
351,BBC Somali,Daawo: Sida gabadhaan Soomaaliyeed uga badbaaday kansarka naasaha,17,Haboon ayaa hadda haweenka ka wacyi galisa kansarka una shegta inaysan ceeb dareemin oo ay la tacaalaan.,https://www.bbc.com/somali/articles/cpw584ny9jwo
367,BBC Somali,Daawo: Muxuu yahay talaalka cusub ee lagula dagaalamayo duumada.,40,"Sanadkii 2021 kii oo kaliya, waxay duumadu ku dishay gudaha Afrika 600,000 oo caruur ah, kuwaas oo ay da'doodu ka yarayd 5 sano. Tijaabo cusub oo talaal ah ...",https://www.bbc.com/somali/articles/cnevyl8gzjjo
428,BBC Somali,Daawo: Maxay Soomaalidu ka aaminsan yihiin bidaarta?,34,"Soomaaliya, waxaa bilo ka hor ka billowday timo-beerashada oo markii hore dibadda loo aadi jiray. Dad badan ayaa tinta beertay taas oo hadal hayn ka dhalisa...",https://www.bbc.com/somali/articles/cpdkpmyvk15o
517,Goobjoog,Wasaaradda Caafimaadka oo Sharaxday Heerka Diyaargarowga Coronaviruses,35,Wasaaradda caafimaadka iyo daryeelka bulshada xukuumadda federaalka Soomaaliya ayaa maanta war ka soo saartay xaaladda cudurka Coronaviruses oo kiisaska ku ...,https://goobjoog.com/2020/03/02/wasaaradda-caafimaadka-oo-sharaxday-heerka-diyaargarowga-coronaviruses/
729,Goobjoog,Wasaaradda Caafimaadka Soomaaliya oo Ka Warbixisay Xaaladihii Ugu Dambeeyey Ee COVID-19,48,Wasiirka wasaaradda caafimaadka iyo daryeelka bulshada xukuumadda federaalka Soomaaliya Dr. Fowsiyo Abiikar Nuur oo maanta shir jaraaid qabatay ayaa ka warb...,https://goobjoog.com/2020/04/20/wasaaradda-caafimaadka-soomaaliya-oo-ka-warbixisay-xaaladihii-ugu-dambeeyey-ee-covid-19/
886,Goobjoog,"Fowsiyo Abiikar Nuur "" Nafaqo-darro Xun Ayaa Dalka Ku Haysata Caruurta Iyo Haweenka Soomaaliyeed""",10,Halkan Ka Daawo Xogta Ka Soo Baxday Wasaarada Caafimaadka Soomaaliya,https://goobjoog.com/2020/08/25/fowsiyo-abiikar-nuur-nafaqo-darro-xun-ayaa-dalka-ku-haysata-caruurta-iyo-haweenka-soomaaliyeed/
891,Goobjoog,Xisbiga Wadeni Ee Somaliland oo Ka Tacsiyeey Geerida Qoraa Baashe Xaaji Xasan,31,"Xisbiga waddani Ee Somaliland ayaa dhambaal Tacsiya udiray ehalada, qaraabada iyo guud ahaan umada Soomaaliyeed ee uu ka baxay qoraa maxamed Baashe Xaaji Xa...",https://goobjoog.com/2020/06/18/xisbiga-wadeni-ee-somaliland-oo-ka-tacsiyeey-geerida-qoraa-baashe-xaaji-xasan/


## 10. mT5 fields

Since the new feature is **category prediction + headline generation**, we do not include the category in the input.

we use:

```text
input_text: analyze somali news article: <body>
target_text: category: caafimaad | headline: <headline>
```

In [43]:
clean_df["input_text"] = "analyze somali news article: " + clean_df["body_clean"]
clean_df["target_text"] = "category: " + clean_df["category"] + " | headline: " + clean_df["headline_clean"]

model_cols = [
    "url", "source", "category", "headline_clean", "body_clean",
    "body_word_count_clean", "headline_word_count", "input_text", "target_text"
]

model_ready = clean_df[~clean_df[flag_cols].any(axis=1)].copy()
print("Rows after removing flagged rows:", model_ready.shape)
model_ready[model_cols].head()

Rows after removing flagged rows: (1134, 20)


,url,source,category,headline_clean,body_clean,body_word_count_clean,headline_word_count,input_text,target_text
0,https://www.bbc.com/somali/articles/cg7ejn4gx07o,BBC Somali,caafimaad,Daraasad: Uurka labaad wuxuu 'si gaar ah' u beddelaa maskaxda haweenka,"Uuurka labaad ayaa sababa in maskaxda hooyada ay siyaabo ""gaar ah"" isu beddesho kuwaasoo ka caawin kara inay diiradda saaraan dareenkooda, sida lagu ogaaday...",546,11,"analyze somali news article: Uuurka labaad ayaa sababa in maskaxda hooyada ay siyaabo ""gaar ah"" isu beddesho kuwaasoo ka caawin kara inay diiradda saaraan d...",category: caafimaad | headline: Daraasad: Uurka labaad wuxuu 'si gaar ah' u beddelaa maskaxda haweenka
1,https://www.bbc.com/somali/articles/c705rww15e3o,BBC Somali,caafimaad,Dayax-gacmeed bani'aadam sida oo 50 sano ka dib dayaxa u kicitimay,"In ka badan konton sano ka dib, NASA hawshii ugu horeysay ee Dayaxa ayey dib u billowday.\nGantaalka loogu talagalay howlgalka ee la yiraahdo Artemis 2 ayaa...",925,11,"analyze somali news article: In ka badan konton sano ka dib, NASA hawshii ugu horeysay ee Dayaxa ayey dib u billowday.\nGantaalka loogu talagalay howlgalka ...",category: caafimaad | headline: Dayax-gacmeed bani'aadam sida oo 50 sano ka dib dayaxa u kicitimay
2,https://www.bbc.com/somali/articles/cly2y5kg3r0o,BBC Somali,caafimaad,Muhiimadda habeenka 29-aad ee Ramadaan,"Sida habeenada kale ee Ramadaanka, habeenka 29-aad ee bisha barakaysan waa mid ka mid ah habeenada sida gaarka ah looga sugo Laylatul Qadr, habeen aad muhii...",586,5,"analyze somali news article: Sida habeenada kale ee Ramadaanka, habeenka 29-aad ee bisha barakaysan waa mid ka mid ah habeenada sida gaarka ah looga sugo La...",category: caafimaad | headline: Muhiimadda habeenka 29-aad ee Ramadaan
3,https://www.bbc.com/somali/articles/c0lj6935l68o,BBC Somali,caafimaad,Muxuu yahay dhagaxa qoslaya ee laga helay Ingiriiska?,"Christine Clark, oo 64 jir ah, ayaa raadinaysay walxo qadiimi ah maalin ka dib ciidii Christmas-ka iyada oo lugaynaysay jasiiradda Lindisfarne ee waqooyiga ...",599,8,"analyze somali news article: Christine Clark, oo 64 jir ah, ayaa raadinaysay walxo qadiimi ah maalin ka dib ciidii Christmas-ka iyada oo lugaynaysay jasiira...",category: caafimaad | headline: Muxuu yahay dhagaxa qoslaya ee laga helay Ingiriiska?
4,https://www.bbc.com/somali/articles/cg5n002r7l0o,BBC Somali,caafimaad,Maxaad ka taqanaa nuucyada suxuurta ugu caansan Afrika?,"Inta lagu jiro bisha barakeysan ee Ramadan, waxaa kordha rabitaanka cunista cuntooyin kala duwan. Dad badanna waxay tixgeliyaan faa'iidada caafimaad iyo sid...",423,8,"analyze somali news article: Inta lagu jiro bisha barakeysan ee Ramadan, waxaa kordha rabitaanka cunista cuntooyin kala duwan. Dad badanna waxay tixgeliyaan...",category: caafimaad | headline: Maxaad ka taqanaa nuucyada suxuurta ugu caansan Afrika?


## 11. Save review and model-ready files

- `caafimaad_cleaned_for_review.xlsx`: includes flags and before/after columns for manual checking.
- `caafimaad_model_ready.csv`: clean training-ready Caafimaad file.

In [44]:
review_cols = [
    "url", "source", "category_original", "category", "headline", "headline_clean",
    "body", "body_clean", "body_word_count_raw", "body_word_count_clean",
    "headline_word_count", *flag_cols, "input_text", "target_text"
]

clean_df[review_cols].to_excel(OUTPUT_REVIEW_FILE, index=False)
model_ready[model_cols].to_csv(OUTPUT_FINAL_FILE, index=False, encoding="utf-8-sig")

print("Saved:", OUTPUT_REVIEW_FILE)
print("Saved:", OUTPUT_FINAL_FILE)

Saved: clean_dataset_for_review\caafimaad_cleaned_for_review.xlsx
Saved: ready_dataset\caafimaad_model_ready.csv
